<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../.."))

from config.spark_config import SparkConfig
from utils.logger import LoggerFactory
from config.io_config import *
from app.platform_app import PlatformApp
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [2]:
# Initialize shared logger (all logs in this run go to the same file: etl_<run_id>.log)
logger = LoggerFactory.setup_logger(name="ETL", log_dir=LOG_DIR)

# Create Spark session with logging enabled (for tracing Spark-related operations)
spark = SparkConfig.create_spark(app_name="Paypal Analytic", logger=logger, use_databricks=True)

# Initialize main application with Spark and logger (used across ETL pipeline)
app = PlatformApp(spark=spark, logger=logger, catalog_name="paypal_analytic")

2026-04-07 18:03:40 | INFO     | ETL | logger.py:113 | Logger initialized | level=DEBUG | file=C:/01_Data/05-data-engineer-bootcamp/03_paypal_databricks/logs\etl_20260401_193104_713773.log
2026-04-07 18:03:42 | INFO     | ETL | spark_config.py:89 | Connected to Databricks via Spark Connect.
2026-04-07 18:03:42 | INFO     | ETL | platform_app.py:44 | Initializing Data Platform...
2026-04-07 18:03:42 | INFO     | ETL | platform_app.py:50 | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [3]:
df_bronze_payer = spark.sql(f"SELECT * FROM {BRONZE_TRANSACTIONS}")
df_bronze_payer.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Transformations

### Trim spaces

In [4]:
# remove those trim values
df_bronze_payer = clean_dataframe(df=df_bronze_payer)
df_bronze_payer.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Select Features

In [5]:
df_silver_payer = df_bronze_payer.select("transaction_id", "transaction_event_code",
                                        "transaction_updated_date", "payer_info", "elton_created_at", "dt", "hour")
df_silver_payer.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------------+----------+----+
|transaction_id   |transaction_event_code|transaction_updated_date      |payer_info                                                                                                                                                                                                                                                            |elton_created_at              |dt        |hour|
+-----------------+----------------------+------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------

### Extract Data

In [ ]:
# Define schema for payer_info JSON
payer_schema = StructType([
    StructField("account_id", StringType()),
    StructField("email_address", StringType()),
    StructField("address_status", StringType()),
    StructField("payer_status", StringType()),
    StructField("country_code", StringType()),
    StructField("payer_name", StructType([
        StructField("given_name", StringType()),
        StructField("middle_name", StringType()),
        StructField("surname", StringType()),
        StructField("alternate_full_name", StringType())  # fallback full name
    ]))
])

# Parse JSON -> struct (optimize + avoid re-parse)
df = df_silver_payer.withColumn("payer", F.from_json(F.col("payer_info"), payer_schema))

# Build full name from parts (auto ignore NULL)
name_col = F.concat_ws(" ",
    F.col("payer.payer_name.given_name"),
    F.col("payer.payer_name.middle_name"),
    F.col("payer.payer_name.surname")
)

# Clean name: blank -> NULL
clean_name = F.when(F.trim(name_col) == "", None).otherwise(name_col)

# Fallback name: blank -> NULL
alt_name = F.when(
    F.trim(F.col("payer.payer_name.alternate_full_name")) == "", None
).otherwise(F.col("payer.payer_name.alternate_full_name"))

In [7]:
df_silver_payer = df.select(
    "transaction_id",
    "transaction_event_code",

    # Standardize timestamp
    parse_timestamp(F.col("transaction_updated_date")).alias("transaction_updated_date"),

    # account_id: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("payer.account_id")) == "", None)
         .otherwise(F.col("payer.account_id")),
        F.lit("Unknown")
    ).alias("account_id"),

    # email: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("payer.email_address")) == "", None)
         .otherwise(F.col("payer.email_address")),
        F.lit("Unknown")
    ).alias("email_address"),

    # full_name: name parts -> alt -> "Unknown"
    F.coalesce(clean_name, alt_name, F.lit("Unknown")).alias("full_name"),

    # address_status: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("payer.address_status")) == "", None)
         .otherwise(F.col("payer.address_status")),
        F.lit("Unknown")
    ).alias("address_status"),

    # payer_status: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("payer.payer_status")) == "", None)
         .otherwise(F.col("payer.payer_status")),
        F.lit("Unknown")
    ).alias("payer_status"),

    # country_code: blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("payer.country_code")) == "", None)
         .otherwise(F.col("payer.country_code")),
        F.lit("Unknown")
    ).alias("country_code"),

    # Metadata timestamps
    parse_timestamp(F.col("elton_created_at")).alias("elton_created_at"),
    F.col("dt").cast("date").alias("dt"),
    F.col("hour").cast("int").alias("hour")
) \
.filter(F.col("transaction_id").isNotNull()) \
.withColumn("process_timestamp", F.date_trunc("second", F.current_timestamp()))

# Preview result
df_silver_payer.show(n=10, truncate=False)

+-----------------+----------------------+------------------------+-------------+----------------------------+-------------------+--------------+------------+------------+-------------------+----------+----+-------------------+
|transaction_id   |transaction_event_code|transaction_updated_date|account_id   |email_address               |full_name          |address_status|payer_status|country_code|elton_created_at   |dt        |hour|process_timestamp  |
+-----------------+----------------------+------------------------+-------------+----------------------------+-------------------+--------------+------------+------------+-------------------+----------+----+-------------------+
|8F987681LS7188052|T0006                 |2023-10-01 01:02:44     |RW7EKEG97GXZG|erniefishingboat@hotmail.com|The Blacksmith Shop|Y             |Y           |US          |2024-03-07 02:36:03|2024-03-07|2   |2026-04-07 11:04:21|
|95328985WP2643805|T0006                 |2023-10-01 01:17:17     |DL2W39YNL9TAA|randall

### Duplicates

In [8]:
df_silver_payer_final = dedup(
    df_silver_payer,
    dedup_cols=["transaction_id", "transaction_event_code"],
    order_cols=["transaction_updated_date", "dt", "hour", "elton_created_at"],
    logger=logger
)

2026-04-07 18:04:29 | INFO     | ETL | utils.py:156 | Starting deduplication
2026-04-07 18:04:29 | INFO     | ETL | utils.py:157 | Dedup columns: ['transaction_id', 'transaction_event_code']
2026-04-07 18:04:29 | INFO     | ETL | utils.py:158 | Order columns: ['transaction_updated_date', 'dt', 'hour', 'elton_created_at']
2026-04-07 18:04:29 | INFO     | ETL | utils.py:176 | Order direction (desc): [True, True, True, True]
2026-04-07 18:04:29 | INFO     | ETL | utils.py:177 | Nulls last: True
2026-04-07 18:04:31 | INFO     | ETL | utils.py:184 | Input row count: 4792
2026-04-07 18:04:33 | INFO     | ETL | utils.py:219 | Output row count after dedup: 4002
2026-04-07 18:04:33 | INFO     | ETL | utils.py:220 | Removed duplicate rows: 790
2026-04-07 18:04:33 | INFO     | ETL | utils.py:221 | Deduplication completed


In [9]:
df_silver_payer_final.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_event_code: string (nullable = true)
 |-- transaction_updated_date: timestamp (nullable = true)
 |-- account_id: string (nullable = false)
 |-- email_address: string (nullable = false)
 |-- full_name: string (nullable = false)
 |-- address_status: string (nullable = false)
 |-- payer_status: string (nullable = false)
 |-- country_code: string (nullable = false)
 |-- elton_created_at: timestamp (nullable = true)
 |-- dt: date (nullable = true)
 |-- hour: integer (nullable = true)
 |-- process_timestamp: timestamp (nullable = true)



### Transformed data to Silver Layer

In [11]:
if not spark.catalog.tableExists(SILVER_PATH_DISPUTED_PP01_PAYER):
    logger.info("Silver disputed pp01 payer table not found. Creating new table...")
    df_silver_payer_final.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("append") \
                   .saveAsTable(SILVER_PATH_DISPUTED_PP01_PAYER)
    logger.info("Silver disputed pp01 payer table created successfully")
else:
    logger.info("Silver disputed pp01 payer table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver_payer_final, key_cols=["transaction_id", "transaction_event_code"],
           table=SILVER_TABLE_DISPUTED_PP01_PAYER, cdc="transaction_updated_date",
           name_catalog=app.catalog_name, name_schema=SCHEMA_SILVER, logger=logger)
    logger.info("Upsert completed successfully")

2026-04-07 18:05:07 | INFO     | ETL | 2061337215.py:10 | Silver disputed pp01 payer table exists. Performing upsert...
2026-04-07 18:05:07 | INFO     | ETL | utils.py:315 | Starting UPSERT into paypal_analytic.silver.disputed_pp01_payer
2026-04-07 18:05:19 | INFO     | ETL | utils.py:345 | UPSERT completed successfully: paypal_analytic.silver.disputed_pp01_payer
2026-04-07 18:05:19 | INFO     | ETL | 2061337215.py:14 | Upsert completed successfully


In [12]:
app.stop()

2026-04-07 18:05:34 | INFO     | ETL | platform_app.py:259 | Stopping Spark session...
2026-04-07 18:05:34 | INFO     | ETL | platform_app.py:261 | Spark stopped.
